In [1]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import itertools
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn import metrics as metrics
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv("UNSW_NB15_training-set.csv" ) 


test = pd.read_csv("UNSW_NB15_testing-set.csv" )


In [3]:
y = train["attack_cat"].values
from collections import Counter
Counter(y)

Counter({'Normal': 37000,
         'Reconnaissance': 3496,
         'Backdoor': 583,
         'DoS': 4089,
         'Exploits': 11132,
         'Analysis': 677,
         'Fuzzers': 6062,
         'Worms': 44,
         'Shellcode': 378,
         'Generic': 18871})

In [4]:
y1 = test["attack_cat"].values
from collections import Counter
Counter(y1)

Counter({'Normal': 56000,
         'Backdoor': 1746,
         'Analysis': 2000,
         'Fuzzers': 18184,
         'Shellcode': 1133,
         'Reconnaissance': 10491,
         'Exploits': 33393,
         'DoS': 12264,
         'Worms': 130,
         'Generic': 40000})

In [5]:
from sklearn.preprocessing import LabelEncoder
encodings = dict()
for c in train.columns:
    
    if train[c].dtype == "object":
        encodings[c] = LabelEncoder()
        encodings[c]
        train[c] = encodings[c].fit_transform(train[c])

In [6]:
y = train.pop("attack_cat").values
X = train.values

In [7]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

In [8]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)
principalComponents = pca.fit_transform(X)
principalDfX = pd.DataFrame(data = principalComponents)

In [9]:
principalDfX.head()
print(Counter(y))

Counter({6: 37000, 5: 18871, 3: 11132, 4: 6062, 2: 4089, 7: 3496, 0: 677, 1: 583, 8: 378, 9: 44})


In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(principalDfX, y, test_size=0.2, random_state=42)

In [11]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5,weights='uniform')
neigh.fit(X_train, y_train)


KNeighborsClassifier()

In [12]:
y_pred = neigh.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7433, 5: 3632, 3: 2472, 4: 1272, 2: 798, 7: 646, 0: 144, 1: 43, 8: 26, 9: 1})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [13]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   8    2   22   75   23    0    1    0    0    0]
 [   4    0    7   78   27    0    0    1    0    0]
 [  11   13  311  373   49    3    3   18    5    0]
 [  77   12  341 1558  192   14    5   69    6    1]
 [  43   10   47  172  864    4   12   58    2    0]
 [   0    1   12   70   16 3610    1   13    0    0]
 [   0    0    0    5    2    0 7411    0    0    0]
 [   1    5   55  124   81    0    0  456    1    0]
 [   0    0    3   12   17    1    0   30   12    0]
 [   0    0    0    5    1    0    0    1    0    0]]
Accuracy Score : 0.864152547519281
Report : 
              precision    recall  f1-score   support

           0       0.06      0.06      0.06       131
           1       0.00      0.00      0.00       117
           2       0.39      0.40      0.39       786
           3       0.63      0.68      0.66      2275
           4       0.68      0.71      0.70      1212
           5       0.99      0.97      0.98      3723
           6       1.00   

In [14]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(neigh, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8620208001214605
Report : 
              precision    recall  f1-score   support

           0       0.09      0.10      0.09       546
           1       0.01      0.00      0.00       466
           2       0.39      0.38      0.39      3303
           3       0.62      0.68      0.65      8857
           4       0.65      0.70      0.68      4850
           5       0.99      0.97      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.70      0.65      0.67      2773
           8       0.44      0.12      0.19       303
           9       0.00      0.00      0.00        37

    accuracy                           0.86     65865
   macro avg       0.49      0.46      0.47     65865
weighted avg       0.86      0.86      0.86     65865



In [15]:
#SVM
from sklearn.svm import SVC

clf = SVC(gamma='auto',decision_function_shape='ovo')
clf.fit(X_train, y_train)

SVC(decision_function_shape='ovo', gamma='auto')

In [16]:
y_pred=clf.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7427, 5: 3604, 3: 2132, 4: 1650, 2: 963, 7: 688, 8: 3})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [17]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   0    0   28   34   62    0    4    3    0    0]
 [   0    0    6   37   69    0    0    5    0    0]
 [   0    0  425  208  103    2    5   42    1    0]
 [   0    0  371 1539  276    3    8   77    1    0]
 [   0    0   48  108  986    2    6   62    0    0]
 [   0    0    5   82   26 3595    0   15    0    0]
 [   0    0    2   10    2    0 7404    0    0    0]
 [   0    0   74  103  105    2    0  439    0    0]
 [   0    0    4    5   20    0    0   45    1    0]
 [   0    0    0    6    1    0    0    0    0    0]]
Accuracy Score : 0.8738082225056173
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.54      0.49       786
           3       0.72      0.68      0.70      2275
           4       0.60      0.81      0.69      1212
           5       1.00      0.97      0.98      3723
           6       1.00  

In [18]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8704015789873225
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.44      0.49      0.47      3303
           3       0.70      0.67      0.68      8857
           4       0.59      0.81      0.69      4850
           5       1.00      0.96      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.62      0.63      0.63      2773
           8       0.33      0.00      0.01       303
           9       0.00      0.00      0.00        37

    accuracy                           0.87     65865
   macro avg       0.47      0.46      0.44     65865
weighted avg       0.86      0.87      0.87     65865



In [19]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0,solver='saga',multi_class='multinomial').fit(X_train, y_train)

In [20]:
y_pred = clf.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7875, 5: 4601, 3: 2058, 4: 926, 7: 567, 2: 440})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [21]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Accuracy Score : 0.8056112224448898
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.25      0.32       786
           3       0.62      0.56      0.59      2275
           4       0.65      0.50      0.57      1212
           5       0.78      0.96      0.86      3723
           6       0.92      0.98      0.95      7418
           7       0.62      0.49      0.55       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.81     16467
   macro avg       0.40      0.37      0.38     16467
weighted avg       0.77      0.81      0.78     16467



In [22]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.7990283154938131
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.41      0.21      0.27      3303
           3       0.60      0.56      0.58      8857
           4       0.62      0.45      0.52      4850
           5       0.77      0.96      0.85     15148
           6       0.92      0.98      0.95     29582
           7       0.61      0.47      0.53      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.80     65865
   macro avg       0.39      0.36      0.37     65865
weighted avg       0.76      0.80      0.78     65865



In [23]:
#Multi Layer Perceptron
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(alpha=1)
clf.fit(X_train, y_train)
clf

MLPClassifier(alpha=1)

In [24]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   0    0   19    9    0   58   41    4    0    0]
 [   0    0    5   22    5   55   27    3    0    0]
 [   0    0  195  182   26  278   66   39    0    0]
 [   0    0  165 1272  119  340  317   62    0    0]
 [   0    0   26  209  606  187  138   46    0    0]
 [   0    0    1   79   18 3582   21   22    0    0]
 [   0    0    0   89   59   12 7258    0    0    0]
 [   0    0   29  183   75   80    3  353    0    0]
 [   0    0    0    7   17    9    4   38    0    0]
 [   0    0    0    6    1    0    0    0    0    0]]
Accuracy Score : 0.8056112224448898
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.25      0.32       786
           3       0.62      0.56      0.59      2275
           4       0.65      0.50      0.57      1212
           5       0.78      0.96      0.86      3723
           6       0.92  

In [25]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8512108099901313
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.43      0.35      0.38      3303
           3       0.65      0.67      0.66      8857
           4       0.56      0.73      0.63      4850
           5       0.94      0.96      0.95     15148
           6       0.99      0.99      0.99     29582
           7       0.57      0.51      0.54      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.85     65865
   macro avg       0.41      0.42      0.42     65865
weighted avg       0.84      0.85      0.84     65865



In [27]:
#RandomForest
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor()
_ = forest.fit(X_train, y_train)
arr = forest.predict(X_train).astype(int)
print(classification_report(y_train, arr))

              precision    recall  f1-score   support

           0       0.88      0.75      0.81       546
           1       0.56      0.80      0.66       466
           2       0.42      0.89      0.57      3303
           3       0.67      0.54      0.60      8857
           4       0.74      0.54      0.62      4850
           5       0.92      0.95      0.93     15148
           6       0.94      0.97      0.96     29582
           7       0.78      0.22      0.34      2773
           8       1.00      0.00      0.01       303
           9       0.00      0.00      0.00        37

    accuracy                           0.83     65865
   macro avg       0.69      0.57      0.55     65865
weighted avg       0.85      0.83      0.82     65865



In [28]:
arr = forest.predict(X_test).astype(int)
print(classification_report(y_test, arr))

              precision    recall  f1-score   support

           0       0.05      0.02      0.02       131
           1       0.16      0.33      0.22       117
           2       0.27      0.60      0.38       786
           3       0.53      0.40      0.46      2275
           4       0.50      0.47      0.48      1212
           5       0.86      0.93      0.89      3723
           6       0.96      0.96      0.96      7418
           7       0.88      0.17      0.29       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.77     16467
   macro avg       0.42      0.39      0.37     16467
weighted avg       0.79      0.77      0.77     16467



In [29]:
#NaiveBayes
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, y_train)
y_pred  =  classifier.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.06      0.19      0.10       546
           1       0.04      0.59      0.08       466
           2       0.11      0.02      0.03      3303
           3       0.64      0.23      0.34      8857
           4       0.13      0.16      0.14      4850
           5       0.89      0.66      0.76     15148
           6       0.93      0.62      0.74     29582
           7       0.13      0.80      0.22      2773
           8       0.01      0.00      0.00       303
           9       0.01      0.05      0.01        37

    accuracy                           0.51     65865
   macro avg       0.30      0.33      0.24     65865
weighted avg       0.73      0.51      0.58     65865



In [30]:
y_pred  =  classifier.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.07      0.22      0.11       131
           1       0.04      0.57      0.07       117
           2       0.13      0.02      0.04       786
           3       0.70      0.25      0.36      2275
           4       0.14      0.16      0.15      1212
           5       0.89      0.64      0.75      3723
           6       0.94      0.63      0.75      7418
           7       0.12      0.75      0.21       723
           8       0.00      0.00      0.00        75
           9       0.01      0.14      0.02         7

    accuracy                           0.51     16467
   macro avg       0.30      0.34      0.25     16467
weighted avg       0.74      0.51      0.58     16467



In [31]:
#DecisionTree Entropy
from sklearn.tree import DecisionTreeClassifier
clf_entropy = DecisionTreeClassifier(criterion = "entropy", random_state = 100,max_depth = 3, min_samples_leaf = 5)
  
clf_entropy.fit(X_train, y_train)
y_pred = clf_entropy.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.37      0.72      0.49      8857
           4       0.00      0.00      0.00      4850
           5       0.83      0.92      0.87     15148
           6       0.91      0.98      0.94     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.75     65865
   macro avg       0.21      0.26      0.23     65865
weighted avg       0.65      0.75      0.69     65865



In [32]:
y_pred = clf_entropy.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.37      0.71      0.49      2275
           4       0.00      0.00      0.00      1212
           5       0.84      0.92      0.88      3723
           6       0.91      0.98      0.94      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.75     16467
   macro avg       0.21      0.26      0.23     16467
weighted avg       0.65      0.75      0.69     16467

